# TradeSense · Component 03 · notebook 32: the one-time TEST opening of the bar models and fusion v0 (CPU or GPU, about 10 to 25 minutes)

**What this notebook does.** It opens TEST-B (genuine 2024-09-01..10-31, 61 days) and TEST-3 (synthetic events on those bars, 300 per symbol, 8 families) **once**, with everything frozen before: the three final bar models (AE-ctx with the A6 aggregation, XGBoost v2, Transformer v2), the thresholds (T1c, q per budget from genuine DEV bars), the fusion v0 stacker and the Watch / Alert tiers. It then reports the numbers as percentages, whatever they are.

**Order of work (nothing about TEST is looked at before step 3).**
1. Load the frozen weights through a small inference library (`c03_infer.py`, the same file the live engine will use).
2. **Parity gate.** Re-score the already-seen genuine August and CAL-3 replica 0 with the library and compare with the scores saved by notebooks 28-31, 35, 41. If anything differs the notebook stops and TEST stays closed.
3. Score TEST-B and TEST-3, save the scores, evaluate with the pre-registered rules below, write `test_opened_v1.json`. If that file exists, the notebook refuses to run again and shows the saved report.

**Pre-registered hypotheses (TEST-3, T1c, primary budget 2 genuine episodes/day, mean of BTC/ETH, macro over the eight families unless stated; each is reported as failed if it fails).**
- **H-T1 (burden holds):** realised genuine TEST-B episodes/day at budget 2 lie in [1, 4] (0.5x to 2x the budget) for fusion and for each of the three models.
- **H-T2 (carry-over):** fusion macro-8 recall on TEST-3 is not more than 0.05 below its CAL-3 value (`fusion_v0_cal3_table.csv`).
- **H-T3 (fusion beats the best single model):** fusion macro-8 minus the macro-8 of the strongest single model (chosen on TEST itself, which is the harder comparison for fusion) is at least +0.03 and the paired 95% event-bootstrap interval of the difference is above 0.
- **H-T4 (burden versus the union):** fusion recall on the six old families is not more than 0.05 below the union of the three models (each at its own 2/day rule), and the fusion genuine episodes/day are at most 60% of the union's.
- **H-T5 (unseen families):** fusion recall on momentum_ignition and slow_accumulation is not more than 0.05 below the best single model on those two.
- **H-T6 (ranking quality):** fusion PR-AUC on replica 0 (positives: event bars of all 8 families; negatives: clean bars with no event in the previous 16 bars) is not more than 0.05 below the same quantity on CAL-3.
- **Descriptive, no decision:** recall at budgets 1 and 4, per family, Watch-4 / Watch-8 tiers and their genuine cost, September versus October burden, per-symbol results.
- **Not decided here:** H-B6 (v2 beats the v1 models re-scored). The v1 models are re-scored by a separate notebook (32b) on the TEST-3 events and scores saved here; the v2 models are not touched again.

**No change after the numbers.** A failed hypothesis is reported as failed. No model, normalisation, q, stacker or tier is retuned on TEST; any improvement becomes a new version evaluated on later, unseen data. TGNN test days are separate (protocol F1/D3).

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime, glob, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
import torch, joblib
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; FUS = OUT/'fusion'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
GUARD = OUT/'test_opened_v1.json'
if GUARD.exists():
    g_ = json.loads(GUARD.read_text()); sb_ = OUT/'b2_step17_share_block.txt'
    if sb_.exists(): print(sb_.read_text())
    raise RuntimeError(f"TEST-B / TEST-3 were already opened on {g_.get('opened_utc')} (see test_opened_v1.json). The numbers above are the saved report. This notebook will not score TEST again.")
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'fusion_v0_cal3_table.csv', 'fusion_v0_cal3_by_family.csv', 'b2_lib.py']:
    assert (OUT/f).exists(), f'{f} missing: run notebooks 31, 41, 35, 40 to the end first'
assert (FUS/'fusion_v0_stacker.joblib').exists()
for s in SYMBOLS: assert (OUT/'scores'/f'fusion_v0_{s}.npz').exists() and (OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz').exists()
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
PROTO_T = r"""# TradeSense C03 - Protocol T0 (TEST-B / TEST-3 opening of the bar models and fusion v0) - 2026-10-15
Written BEFORE TEST-B / TEST-3 scores exist. Frozen: protocol B2 (and A1-A4, A6), declared_final_set_v1.json, declared_ae_v2.json, declared_fusion_v0.json, declared_watch_v1.json, the stacker file, all weight files.
Rules: T1c at budgets 1/2/4 (primary 2), q fixed from genuine DEV; fusion stacker fixed; Watch-4 / Watch-8 q fixed by notebook 40. Hypotheses H-T1..H-T6 as written in notebook 32 (burden in [1,4]; carry-over within 0.05 of CAL-3; fusion beats the strongest single model by >= 0.03 with paired CI above 0; fusion within 0.05 of the union on old families at <= 60% of its burden; unseen families within 0.05 of the best single; PR-AUC within 0.05 of CAL-3).
Opened once; scores saved; no retuning; failed hypotheses reported as failed; H-B6 deferred to notebook 32b (v1 re-scoring on the saved TEST-3 events).
"""
PH = hashlib.sha256(PROTO_T.encode()).hexdigest(); (FUS/'protocol_test_b_v1.md').write_text(PROTO_T)
say('protocol B2 intact, sha256 ' + ph[:16] + '; test protocol T0 sha256 ' + PH[:16] + '; device ' + DEVICE + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; test protocol T0 sha256 876d176c01c0712a; device cpu


## 1 · Inference library

In [ ]:
open('c03_infer.py', 'w').write(r'''"""TradeSense C03 - inference library for the frozen final set (Protocol B2 + amendment A6 + fusion v0).

Pure inference: no training, no labels. The same file is used by notebook 32 (TEST opening, batch) and by the engine (live, on a trailing buffer).
Every function reproduces, bar for bar, what notebooks 28-31, 35 and 41 did when they saved the scores; notebook 32 proves this with a parity gate before TEST is touched.
"""
import json, numpy as np, pandas as pd, torch, torch.nn as nn, xgboost as xgb
from pathlib import Path
import b2_lib as B

FEATS = B.FEATS; WARM = B.WARM
MODEL_NAMES = ['AE-ctx fmax', 'XGBoost v2', 'Transformer v2']
SEEDS = [11, 22, 33]


def nanw(a):
    a = np.asarray(a, 'float64').copy(); a[:WARM] = np.nan; return a


# ------------------------------------------------------------------ XGBoost features (53 causal rolling features of the 6 normalised features)
def build_feats(Z):
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:WARM] = np.nan; return out, list(cols)


FA_NAMES = build_feats(np.zeros((200, 6)))[1]


# ------------------------------------------------------------------ Transformer
class TNet(nn.Module):
    def __init__(s, W, d, L, heads=4, drop=0.1):
        super().__init__(); s.inp = nn.Linear(6, d); s.pos = nn.Parameter(torch.zeros(1, W, d)); nn.init.normal_(s.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, batch_first=True, norm_first=True, activation='gelu')
        s.enc = nn.TransformerEncoder(layer, L, enable_nested_tensor=False); s.norm = nn.LayerNorm(d); s.out = nn.Linear(d, 1)

    def forward(s, x): return s.out(s.norm(s.enc(s.inp(x) + s.pos)[:, -1])).squeeze(-1)


@torch.no_grad()
def tr_score_series(net, z, W, device='cpu', bs=8192):
    net.eval(); Zt = torch.tensor(np.asarray(z, 'float32'), dtype=torch.float32, device=device); n = len(z); out = np.full(n, np.nan, dtype=np.float32); lo = max(WARM, W - 1)
    ar = torch.arange(-W + 1, 1, device=device)
    for a in range(lo, n, bs):
        gi = torch.arange(a, min(n, a + bs), device=device); out[a:a + len(gi)] = net(Zt[gi[:, None] + ar[None, :]]).float().cpu().numpy()
    return out


# ------------------------------------------------------------------ AE-ctx with the A6 aggregation (mean of the last 10 per-bar fmax values)
def ae_postprocess(cols8, k=10):
    """cols8 = B.ae_columns(...) (float32, 8 columns). Returns the stored A6 score: causal mean of fmax over k bars, NaN->0 (as saved by notebook 41), first WARM bars NaN."""
    f = nanw(cols8[:, 0]); v = pd.Series(f).rolling(k, min_periods=k).mean().to_numpy()
    return nanw(np.nan_to_num(v, nan=0.0).astype('float32'))


class Bundle:
    """The three frozen bar models of one symbol. `modes` gives the normalisation of each model ('N0' / 'N1')."""
    def __init__(self, sym, n0, modes, ae_nets, ae_scale, xgb_boosters, tr_nets, tr_W, ae_k=10, device='cpu'):
        self.sym, self.n0, self.modes, self.ae_nets, self.ae_scale, self.xgb, self.tr_nets, self.tr_W, self.ae_k, self.device = sym, n0, modes, ae_nets, np.asarray(ae_scale), xgb_boosters, tr_nets, tr_W, ae_k, device

    def score(self, F, want_cols=False):
        """F: DataFrame/array with the 6 Layer 1 features of one continuous 1-minute series (already with the contract's warm-up fix). Returns {model name: float64 scores, NaN before bar 30}."""
        zs = {m: B.normalise(F, m, self.n0) for m in sorted(set(self.modes.values()))}; out = {}
        err = np.mean([B.ae_errors(n, zs[self.modes[MODEL_NAMES[0]]]) for n in self.ae_nets], axis=0); cols = B.ae_columns(err, self.ae_scale).astype('float32')
        out[MODEL_NAMES[0]] = nanw(ae_postprocess(cols, self.ae_k))
        X = build_feats(zs[self.modes[MODEL_NAMES[1]]])[0]; dm = xgb.DMatrix(X, feature_names=FA_NAMES)
        m = np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in self.xgb], axis=0).astype(np.float32); m[:WARM] = np.nan; out[MODEL_NAMES[1]] = nanw(m)
        z = zs[self.modes[MODEL_NAMES[2]]].astype(np.float32); t = np.mean([tr_score_series(n_, z, self.tr_W, self.device) for n_ in self.tr_nets], axis=0).astype(np.float32); t[:max(WARM, self.tr_W - 1)] = np.nan
        out[MODEL_NAMES[2]] = nanw(t)
        if want_cols: out['_ae_cols'] = cols
        return out


def load_bundle_from_training(out_dir, sym, decl, ae_scale, n0, device='cpu'):
    """Loads the weights exactly where notebooks 28-30 saved them (models_b2_v1/ae, /xgb, /transformer)."""
    out_dir = Path(out_dir); modes = {m: decl['models'][m]['normalisation'] for m in MODEL_NAMES}
    ae_nets = []
    for sd in SEEDS:
        ck = torch.load(out_dir / 'ae' / f'{sym}_{modes[MODEL_NAMES[0]]}_final_s{sd}.pt', map_location='cpu'); n = B.AECtx(); n.load_state_dict(ck['state']); n.eval(); ae_nets.append(n)
    cx = decl['models']['XGBoost v2']; fs, scope, dtxt = cx['configuration'].split('|'); depth = int(dtxt[1:]); key = 'pool' if scope == 'pooled' else sym; xb = []
    assert fs == 'A', 'the library implements feature set A (53 features) only'
    for sd in cx['seeds']:
        p = out_dir / 'xgb' / f'{modes[MODEL_NAMES[1]]}_{fs}_{scope}_{key}_d{depth}_s{sd}.json'; b_ = xgb.Booster(); b_.load_model(str(p))
        xb.append((b_, json.loads(Path(str(p)[:-5] + '.info.json').read_text())['best_iteration']))
    ct = decl['models']['Transformer v2']; cfg = ct['configuration']; W, d, L = [int(x[1:]) for x in cfg.split('_')]; nets = []
    for sd in ct['seeds']:
        n = TNet(W, d, L); n.load_state_dict(torch.load(out_dir / 'transformer' / f'{modes[MODEL_NAMES[2]]}_{cfg}_s{sd}.pt', map_location='cpu')); n.eval(); nets.append(n.to(device))
    return Bundle(sym, n0, modes, ae_nets, ae_scale, xb, nets, W, device=device)


# ------------------------------------------------------------------ fusion v0 (Integrity Score input): causal rolling 7-day robust z of each model score -> 8 features -> logistic stacker
TRF = {'AE-ctx fmax': lambda x: np.log1p(np.maximum(x, 0)), 'XGBoost v2': lambda x: x, 'Transformer v2': lambda x: x}
FNAMES = ['z ' + m for m in MODEL_NAMES] + ['z max', 'z mean'] + ['max5 ' + m for m in MODEL_NAMES]


def gen_stats(gen_l1_scores, valid):
    """DEV genuine median / scale of every model (fallback while history is short). gen_l1_scores: {model: saved GEN_L1 score}; valid: bool mask of the DEV bars."""
    out = {}
    for m in MODEL_NAMES:
        g = TRF[m](gen_l1_scores[m])[valid]; out[m] = (float(np.nanmedian(g)), float((np.nanpercentile(g, 75) - np.nanpercentile(g, 25)) / 1.349))
    return out


def ref_stats(ref_scores, gstats):
    """rolling statistics of the GENUINE series (T1c idea: dense synthetic events must not contaminate the history)."""
    out = {}
    for m in MODEL_NAMES:
        x = TRF[m](ref_scores[m]); r = pd.Series(x).rolling(10080, min_periods=1440)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy(); gmed, gsc = gstats[m]
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * gsc); bad = np.isnan(med) | np.isnan(scl); out[m] = (np.where(bad, gmed, med), np.where(bad, gsc, scl))
    return out


def fusion_features(scores, rstats):
    Zs = np.column_stack([np.clip((TRF[m](scores[m]) - rstats[m][0]) / rstats[m][1], -15, 15) for m in MODEL_NAMES])
    roll = np.column_stack([pd.Series(Zs[:, i]).rolling(5, min_periods=1).max().to_numpy() for i in range(len(MODEL_NAMES))])
    return np.column_stack([Zs, Zs.max(axis=1), Zs.mean(axis=1), roll]).astype(np.float32)


def fusion_margin(stacker, X):
    ok = ~np.isnan(X).any(axis=1); o = np.full(len(X), np.nan)
    if ok.any(): o[ok] = stacker.decision_function(X[ok]) if hasattr(stacker, 'decision_function') else stacker.predict(X[ok], output_margin=True)
    return o


def fusion_contributions(stacker, X):
    """exact additive contribution of every input to the logit (logistic stacker): (x - mean) * coef / scale."""
    sc, lr = stacker.steps[0][1], stacker.steps[1][1]; w = lr.coef_[0] / sc.scale_; return (X - sc.mean_[None, :]) * w[None, :]


def fusion_score(scores, ref_scores, gstats, stacker):
    return fusion_margin(stacker, fusion_features(scores, ref_stats(ref_scores, gstats)))


def rolling_threshold(genuine_score, q):
    t = B.rolling_thr(genuine_score, q); return np.where(np.isnan(t), np.inf, t)
''')

_old = (OUT/'c03_infer.py').read_text() if (OUT/'c03_infer.py').exists() else None
if _old is not None and _old.strip() != Path('c03_infer.py').read_text().strip(): say('NOTE: models_b2_v1/c03_infer.py differed from this notebook copy; the notebook copy is used and written back.')
(OUT/'c03_infer.py').write_text(Path('c03_infer.py').read_text()); sys.path.insert(0, '.'); import c03_infer as c; importlib.reload(c)
say('inference library sha256 ' + sha_file('c03_infer.py') + '; b2_lib sha256 ' + sha_file(OUT/'b2_lib.py'))

inference library sha256 3b5d9df23521adf9; b2_lib sha256 1b0b46630763c944


## 2 · Frozen weights, saved scores

In [ ]:
AOLEN = 31 * 1440; AOFULL = 132480; TEST_LO = AOLEN; CAL_LO = 1440; DEV_LO, DEV_HI = 90720, 129600
L1V = B.valid_mask(129600, DEV_LO, DEV_HI); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN); TV = np.zeros(AOFULL, bool); TV[TEST_LO:] = True; TV[:B.WARM] = False
DF = json.load(open(OUT/'declared_final_set_v1.json')); AE2 = json.load(open(OUT/'declared_ae_v2.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json')); WATCH = json.load(open(OUT/'declared_watch_v1.json'))
MODELS = c.MODEL_NAMES; assert list(DF['models']) == MODELS and AE2['variant'] == 'mean10' and AE2['normalisation'] == 'N1' and DF['models'][MODELS[0]]['normalisation'] == 'N1'
MODES = {m: DF['models'][m]['normalisation'] for m in MODELS}; assert MODES == FUSD['input_models'], (MODES, FUSD['input_models'])
FILES = {MODELS[0]: 'ae_ctx_agg', MODELS[1]: 'xgb_v2', MODELS[2]: 'transformer_v2'}
assert FUSD['inputs'] == c.FNAMES
STACKER = joblib.load(FUS/'fusion_v0_stacker.joblib')
if FUSD['stacker']['kind'] == 'LR':
    lg = FUSD['logistic']; assert np.allclose(STACKER.steps[1][1].coef_[0], lg['coef']) and np.allclose(STACKER.steps[0][1].mean_, lg['scaler_mean']), 'stacker file differs from the declared one'
def find1(pat):
    c_ = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c_, pat; return sorted(c_)[0]
AOF = {s: BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet' for s in SYMBOLS}
N0 = {}; SAVED = {}; AEC = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    N0[s] = B.fit_n0(a[B.FEATS].reset_index(drop=True).iloc[15:90720])
    AEC[s] = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz')
    for m in MODELS:
        z = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{MODES[m]}.npz'); assert not any('TEST' in k for k in z.files)
        for k in ['GEN_L1', 'GEN_AO', 'CAL_r0']:
            v = z[k]; v = v[:, 0] if v.ndim == 2 else v; SAVED[(m, s, k)] = c.nanw(v[:AOLEN] if k != 'GEN_L1' else v)
    zf = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in zf.files)
    SAVED[('Fusion', s, 'GEN_AO')] = zf['GEN_AO']; SAVED[('Fusion', s, 'CAL_r0')] = zf['CAL_r0']
BUN = {s: c.load_bundle_from_training(OUT, s, DF, AEC[s]['scale'], N0[s], DEVICE) for s in SYMBOLS}
GST = {s: c.gen_stats({m: SAVED[(m, s, 'GEN_L1')] for m in MODELS}, L1V) for s in SYMBOLS}
say('frozen weights loaded: AE-ctx (3 seeds, N1, mean10), XGBoost ' + DF['models'][MODELS[1]]['configuration'] + ' (' + MODES[MODELS[1]] + '), Transformer ' + DF['models'][MODELS[2]]['configuration'] + ' (' + MODES[MODELS[2]] + '); stacker ' + FUSD['stacker']['kind'] + ' C=' + str(FUSD['stacker']['C']))

frozen weights loaded: AE-ctx (3 seeds, N1, mean10), XGBoost A|pooled|d3 (N0), Transformer W16_d32_L1 (N1); stacker LR C=0.01


## 3 · Parity gate (TEST stays closed until it passes)

In [ ]:
# frozen inputs and the three TEST-independent checks. Only genuine August (bars < 44640) and CAL-3 replica 0 are used here; TEST-B bars are not scored yet.
PAR = {}; RES = []
for s in SYMBOLS:
    ao = pd.read_parquet(AOF[s], columns=B.FEATS).iloc[:AOLEN].reset_index(drop=True); ao.loc[:14, 'volume_acceleration'] = 0.0
    cf = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet'); cf = cf[cf.replica == 0].iloc[:AOLEN].reset_index(drop=True)
    PAR[s] = {'GEN_AO': ao, 'CAL_r0': cf}
def cmp(a, b, name):
    fa, fb = np.isfinite(a), np.isfinite(b); same_nan = bool((fa == fb).all()); m = fa & fb; d = np.abs(a[m] - b[m]); rho = spearmanr(a[m][::7], b[m][::7])[0]
    return same_nan, float(d.max()), float(np.median(d)), float(rho)
SCP = {}
for s in SYMBOLS:
    for k in ['GEN_AO', 'CAL_r0']:
        SCP[(s, k)] = BUN[s].score(PAR[s][k], want_cols=(k == 'GEN_AO'))
    for m in MODELS:
        for k in ['GEN_AO', 'CAL_r0']:
            same, mx, md, rho = cmp(SCP[(s, k)][m], SAVED[(m, s, k)], m); RES.append({'sym': s, 'model': m, 'series': k, 'same NaN pattern': same, 'max abs diff': mx, 'median abs diff': md, 'spearman': rho})
RES = pd.DataFrame(RES); say('--- parity: library re-scoring versus the scores saved by notebooks 28-31 / 41 (August genuine and CAL-3 replica 0) ---'); say(RES.to_string(index=False, float_format=lambda x: f'{x:.2e}' if abs(x) < 0.01 else f'{x:.5f}'))
tol = {MODELS[0]: 2e-3, MODELS[1]: 2e-3, MODELS[2]: 2e-2}      # Transformer: saved on GPU, re-scored here on a different device / kernels (float32): differences of ~1e-3 are expected
for m in MODELS:
    r_ = RES[RES.model == m]; assert r_['same NaN pattern'].all() and (r_['max abs diff'] <= tol[m]).all() and (r_.spearman > 0.9999).all(), f'PARITY FAILED for {m}: TEST stays closed'
say('PASS T01 re-scoring reproduces the saved scores of all three models (same NaN pattern, max abs diff within tolerance, Spearman > 0.9999)')
QP = {MODELS[0]: AE2['q_T1_by_budget'], MODELS[1]: DF['models'][MODELS[1]]['q_T1_by_budget'], MODELS[2]: DF['models'][MODELS[2]]['q_T1_by_budget']}; AGR = []
for s in SYMBOLS:
    for m in MODELS:
        q = QP[m]['2'][s]; thr_o = c.rolling_threshold(SAVED[(m, s, 'GEN_AO')], q); thr_n = c.rolling_threshold(SCP[(s, 'GEN_AO')][m], q)
        for k in ['GEN_AO', 'CAL_r0']:
            ao_, an_ = SAVED[(m, s, k)] > thr_o, SCP[(s, k)][m] > thr_n; un = int((ao_ | an_).sum()); AGR.append({'sym': s, 'model': m, 'series': k, 'alarm bars saved': int(ao_.sum()), 'alarm bars now': int(an_.sum()), 'bars that differ': int((ao_ ^ an_).sum()), 'episodes saved': B.episodes(ao_), 'episodes now': B.episodes(an_)})
AGR = pd.DataFrame(AGR); say(''); say('--- the decision that matters: alarms at the frozen budget-2 rule, saved scores versus re-scored ---'); say(AGR.to_string(index=False))
assert (AGR['bars that differ'] <= np.maximum(2, 0.01 * AGR['alarm bars saved'])).all() and ((AGR['episodes saved'] - AGR['episodes now']).abs() <= 1).all(), 'PARITY FAILED: alarms differ between saved and re-scored scores; TEST stays closed'
say('PASS T01b alarms at the frozen budget-2 rule are the same (at most 1% of alarm bars differ, episodes within 1) for all models, both series')
for s in SYMBOLS:       # the A6 aggregation applied to the saved AE-ctx error columns reproduces the saved aggregated score exactly
    pp = c.ae_postprocess(AEC[s]['GEN_AO'][:AOLEN]); sv = np.load(OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz')['GEN_AO'][:AOLEN, 0]; assert np.allclose(np.nan_to_num(pp, nan=-1), np.nan_to_num(c.nanw(sv), nan=-1), atol=1e-6), s
say('PASS T02 the A6 aggregation (mean of 10 fmax values) applied to the saved AE error columns equals the saved aggregated AE score')
for s in SYMBOLS:       # fusion code versus the saved fused scores, using the SAVED model scores (isolates the fusion code)
    gs = {m: SAVED[(m, s, 'GEN_AO')] for m in MODELS}; rs = c.ref_stats(gs, GST[s])
    for k in ['GEN_AO', 'CAL_r0']:
        fz = c.fusion_margin(STACKER, c.fusion_features({m: SAVED[(m, s, k)] for m in MODELS}, rs)); sv = SAVED[('Fusion', s, k)]
        same, mx, md, rho = cmp(fz, sv, 'fusion'); assert same and mx < 1e-4, (s, k, mx)
say('PASS T03 the library fusion code reproduces the saved fused scores from the saved model scores (max abs diff < 1e-4)')
for s in SYMBOLS:       # fusion through the whole chain (re-scored models -> fusion)
    gs = {m: SCP[(s, 'GEN_AO')][m] for m in MODELS}; rs = c.ref_stats(gs, GST[s])
    for k in ['GEN_AO', 'CAL_r0']:
        fz = c.fusion_margin(STACKER, c.fusion_features({m: SCP[(s, k)][m] for m in MODELS}, rs)); sv = SAVED[('Fusion', s, k)]
        same, mx, md, rho = cmp(fz, sv, 'fusion'); assert same and rho > 0.9999 and mx < 1.5e-1, (s, k, mx, rho)
say('PASS T04 end to end (weights -> three scores -> fusion) reproduces the saved fused scores on August and CAL-3 replica 0')
s0 = SYMBOLS[0]; part = BUN[s0].score(PAR[s0]['GEN_AO'].iloc[:20000].reset_index(drop=True)); full = SCP[(s0, 'GEN_AO')]
for m in MODELS: assert np.allclose(np.nan_to_num(part[m], nan=-1), np.nan_to_num(full[m][:20000], nan=-1), atol=2e-2), m
say('PASS T05 causality: scores of the first 20,000 bars do not change when later bars are added')
for q in [FUSD['q_T1_by_budget'][str(b_)] for b_ in B.BUDGETS]: assert set(q) == set(SYMBOLS)
assert WATCH['alert']['q'] == FUSD['q_T1_by_budget']['2'] and WATCH['watch']['q']['4'] == FUSD['q_T1_by_budget']['4'], 'Alert / Watch-4 q differ from the fusion declaration'
say('PASS T06 declared q values: Alert (watch file) equals fusion budget 2, Watch-4 equals fusion budget 4')
say('PARITY GATE PASSED: TEST-B / TEST-3 are opened now.')

--- parity: library re-scoring versus the scores saved by notebooks 28-31 / 41 (August genuine and CAL-3 replica 0) ---
    sym          model series  same NaN pattern  max abs diff  median abs diff  spearman
BTCUSDT    AE-ctx fmax GEN_AO              True      0.00e+00         0.00e+00   1.00000
BTCUSDT    AE-ctx fmax CAL_r0              True      0.00e+00         0.00e+00   1.00000
BTCUSDT     XGBoost v2 GEN_AO              True      0.00e+00         0.00e+00   1.00000
BTCUSDT     XGBoost v2 CAL_r0              True      0.00e+00         0.00e+00   1.00000
BTCUSDT Transformer v2 GEN_AO              True      8.86e-03         8.68e-04   1.00000
BTCUSDT Transformer v2 CAL_r0              True      8.73e-03         8.17e-04   1.00000
ETHUSDT    AE-ctx fmax GEN_AO              True      0.00e+00         0.00e+00   1.00000
ETHUSDT    AE-ctx fmax CAL_r0              True      0.00e+00         0.00e+00   1.00000
ETHUSDT     XGBoost v2 GEN_AO              True      0.00e+00         0.00e+00 

## 4 · Open TEST-B / TEST-3 (once)

In [ ]:
# ---------------- TEST-B / TEST-3 scoring (first time these bars are scored). Scores are saved before anything is evaluated.
t0_ = time.time(); TSC = {}; TLAB = {}; EVT = {}; GENF = {}; FULLSC = {}
for s in SYMBOLS:
    ao = pd.read_parquet(AOF[s], columns=B.FEATS).reset_index(drop=True); assert len(ao) == AOFULL, (s, len(ao)); ao.loc[:14, 'volume_acceleration'] = 0.0
    GENF[s] = BUN[s].score(ao)
    for m in MODELS: assert np.allclose(np.nan_to_num(GENF[s][m][:AOLEN], nan=-1), np.nan_to_num(SCP[(s, 'GEN_AO')][m], nan=-1), atol=2e-2), 'full-series scores must extend the August scores'
    f = pd.read_parquet(L3/f'l3v3_{s}_TEST.parquet'); ev = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); EVT[s] = ev
    for r in sorted(f.replica.unique()):
        fr = f[f.replica == r].reset_index(drop=True); assert len(fr) == AOFULL; TLAB[(s, int(r))] = (fr.label.to_numpy().astype(int), fr.in_seg.to_numpy().astype(bool)); TSC[(s, int(r))] = BUN[s].score(fr[B.FEATS])
    assert ev.t0.min() >= TEST_LO and (ev.t0 + ev.dur).max() <= AOFULL, 'TEST-3 events must lie inside 2024-09-01..10-31'
    rs = c.ref_stats({m: GENF[s][m] for m in MODELS}, GST[s]); FULLSC[(s, 'GEN')] = c.fusion_margin(STACKER, c.fusion_features(GENF[s], rs))
    for (ss, r), sc in TSC.items():
        if ss == s: FULLSC[(s, f'r{r}')] = c.fusion_margin(STACKER, c.fusion_features(sc, rs))
    pack = {}
    for tag, m in zip(['AE', 'XGB', 'TR'], MODELS):
        pack[f'{tag}__GEN'] = GENF[s][m].astype('float32')
        for (ss, r), sc in TSC.items():
            if ss == s: pack[f'{tag}__r{r}'] = sc[m].astype('float32')
    for k, v in FULLSC.items():
        if k[0] == s: pack[f'FUS__{k[1]}'] = v.astype('float32')
    np.savez_compressed(OUT/'scores'/f'test_b_{s}.npz', **pack)
SC = {}
for s in SYMBOLS:
    for m in MODELS:
        SC[(m, s, 'GEN')] = GENF[s][m]
        for (ss, r), sc in TSC.items():
            if ss == s: SC[(m, s, f'r{r}')] = sc[m]
    for k, v in FULLSC.items():
        if k[0] == s: SC[('Fusion', s, k[1])] = v
REPS = {s: sorted(r for (ss, r) in TSC if ss == s) for s in SYMBOLS}
say(f'TEST-B / TEST-3 scored and saved (scores/test_b_BTCUSDT.npz, test_b_ETHUSDT.npz) in {time.time() - t0_:.0f} s; replicas ' + str({s: REPS[s] for s in SYMBOLS}) + '; TEST-3 events per symbol ' + str({s: len(EVT[s]) for s in SYMBOLS}) + '; genuine TEST-B days ' + str(int(TV.sum() // 1440)))

TEST-B / TEST-3 scored and saved (scores/test_b_BTCUSDT.npz, test_b_ETHUSDT.npz) in 115 s; replicas {'BTCUSDT': [0], 'ETHUSDT': [0]}; TEST-3 events per symbol {'BTCUSDT': 300, 'ETHUSDT': 300}; genuine TEST-B days 61


## 5 · Evaluation helpers

In [ ]:
NAMES = MODELS + ['Fusion']; OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW; PRIM = B.PRIMARY; BUD = B.BUDGETS
Q = {MODELS[0]: AE2['q_T1_by_budget'], MODELS[1]: DF['models'][MODELS[1]]['q_T1_by_budget'], MODELS[2]: DF['models'][MODELS[2]]['q_T1_by_budget'], 'Fusion': FUSD['q_T1_by_budget']}
QW = {'2': WATCH['alert']['q'], '4': WATCH['watch']['q']['4'], '8': WATCH['watch']['q']['8']}
THR = {}
def thr(name, s, q):
    k = (name, s, q)
    if k not in THR: THR[k] = c.rolling_threshold(SC[(name, s, 'GEN')], q)
    return THR[k]
def al_gen(name, s, q): return SC[(name, s, 'GEN')] > thr(name, s, q)
def al_inj(name, s, key, q): return SC[(name, s, key)] > thr(name, s, q)
def qof(name, b, s): return Q[name][str(b)][s]
SETS = [(s, r, EVT[s][EVT[s].replica == r].reset_index(drop=True)) for s in SYMBOLS for r in REPS[s]]
META = pd.concat([pd.DataFrame({'sym': s, 'rep': r, 'fam': ev.family.to_numpy(), 't0': ev.t0.to_numpy()}) for s, r, ev in SETS], ignore_index=True)
def event_hits(a, ev):
    cs = np.concatenate([[0], np.cumsum(a.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (cs[t0 + du] - cs[t0]) > 0
HC = {}
def hits(who, b):
    """who: a model name, 'Fusion', 'union' or ('watch', b). Returns the hit vector over all TEST-3 events (order of SETS)."""
    k = (who, b)
    if k not in HC:
        parts = []
        for s, r, ev in SETS:
            if who == 'union': a = np.logical_or.reduce([al_inj(m, s, f'r{r}', qof(m, b, s)) for m in MODELS])
            elif isinstance(who, tuple): a = al_inj('Fusion', s, f'r{r}', QW[str(b)][s])
            else: a = al_inj(who, s, f'r{r}', qof(who, b, s))
            parts.append(event_hits(a, ev))
        HC[k] = np.concatenate(parts)
    return HC[k]
def gen_alarm(who, s, b):
    if who == 'union': return np.logical_or.reduce([al_gen(m, s, qof(m, b, s)) for m in MODELS])
    if isinstance(who, tuple): return al_gen('Fusion', s, QW[str(b)][s])
    return al_gen(who, s, qof(who, b, s))
def gen_eps(who, b, valid=None):
    v = TV if valid is None else valid; return float(np.mean([B.eps_per_day(gen_alarm(who, s, b), v) for s in SYMBOLS]))
def macro_of(h, fams, sel=None, syms=None):
    out = []
    for s in (syms or SYMBOLS):
        k = ((META.sym == s) & META.fam.isin(fams)).to_numpy().copy()
        if sel is not None: k &= sel
        out.append(pd.DataFrame({'fam': META.fam[k].to_numpy(), 'h': h[k]}).groupby('fam').h.mean().mean())
    return float(np.mean(out))
def fam_rates(h):
    return pd.DataFrame({'sym': META.sym, 'fam': META.fam, 'h': h}).groupby(['sym', 'fam']).h.mean().groupby('fam').mean()
GROUPS = [[np.flatnonzero(((META.sym == s) & (META.fam == f)).to_numpy()) for f in ALLF] for s in SYMBOLS]
def boot(h_list, fams, n=None, seed=0):
    """paired event bootstrap (resampling events inside symbol x family); returns (n, len(h_list)) macro values."""
    n = n or NBOOT; rg = np.random.default_rng(seed); out = np.zeros((n, len(h_list))); fi = [ALLF.index(f) for f in fams]
    for i in range(n):
        acc = []
        for gs in GROUPS:
            fm = []
            for j in fi:
                idx = gs[j]; pick = idx[rg.integers(0, len(idx), len(idx))]; fm.append([h[pick].mean() for h in h_list])
            acc.append(np.mean(fm, axis=0))
        out[i] = np.mean(acc, axis=0)
    return out
def ci(x): return f'{100 * np.percentile(x, 2.5):.0f}-{100 * np.percentile(x, 97.5):.0f}'
def chance(who, b, fams, nsh=40, seed=0):
    rg = np.random.default_rng(seed); out = []
    for _ in range(nsh):
        hs = []
        for s, r, ev in SETS:
            seg = gen_alarm(who, s, b)[TEST_LO:]; full = np.zeros(AOFULL, bool); full[TEST_LO:] = np.roll(seg, int(rg.integers(0, len(seg)))); hs.append(event_hits(full, ev))
        out.append(macro_of(np.concatenate(hs), fams))
    return float(np.mean(out))
def pct(x): return f'{100 * x:.1f}%'
# AUCs on replica 0: positives = event bars of the 8 families, negatives = clean bars with no event in the previous 16 bars
def aucs(score_of, lab_of, lo, hi):
    rows = []
    for s in SYMBOLS:
        lab, inseg = lab_of(s); sc = score_of(s); n = len(lab); idx = np.arange(n); cs_ = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
        clean = ((cs_[idx + 1] - cs_[np.maximum(idx - 16, 0)]) == 0) & inseg; pos = (lab > 0) & inseg; ok = np.isfinite(sc) & (idx >= max(B.WARM, lo)) & (idx < hi); m = ok & (pos | clean)
        rows.append((average_precision_score(pos[m], sc[m]), roc_auc_score(pos[m], sc[m])))
    return np.mean(rows, axis=0)
CALLAB = {s: (lambda f: (f.label.to_numpy().astype(int)[:AOLEN], f.in_seg.to_numpy().astype(bool)[:AOLEN]))(pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet', columns=['label', 'in_seg', 'replica']).query('replica == 0')) for s in SYMBOLS}
say('evaluation helpers ready (T1c: the rolling threshold of the genuine series is applied to the injected series; q per symbol and budget frozen)')

evaluation helpers ready (T1c: the rolling threshold of the genuine series is applied to the injected series; q per symbol and budget frozen)


## 6 · Unit tests

In [ ]:
# unit checks on the evaluation code (independent plain-loop recomputation)
s0, r0, ev0 = SETS[0]; a0 = al_inj('Fusion', s0, f'r{r0}', qof('Fusion', PRIM, s0)); h0 = event_hits(a0, ev0)
plain = np.array([bool(a0[t:t + d].any()) for t, d in zip(ev0.t0.to_numpy(), ev0.dur.to_numpy())]); assert (h0 == plain).all()
say('PASS T07 event hits equal a plain loop (first TEST-3 set, fusion, budget 2)')
cq = gen_eps('Fusion', PRIM); g_ = [al_gen('Fusion', s, qof('Fusion', PRIM, s)) for s in SYMBOLS]
def eps_loop(a, v):
    a = a & v; n = 0; last = -10**9
    for i in np.flatnonzero(a):
        if i - last > B.GAPM: n += 1
        last = i
    return n / (v.sum() / 1440.0)
assert abs(np.mean([eps_loop(a, TV) for a in g_]) - cq) < 1e-9
say('PASS T08 genuine episodes/day equal a plain loop (fusion, budget 2)')
for s in SYMBOLS:
    for m in NAMES: assert al_gen(m, s, qof(m, PRIM, s))[:TEST_LO].sum() >= 0
    assert np.isinf(thr('Fusion', s, qof('Fusion', PRIM, s))[:1440]).all(), 'no threshold before 1,440 bars of history'
say('PASS T09 rolling thresholds are undefined (no alarm) before 1,440 bars of history; thresholds use the genuine series only')
fam_ct = META.groupby(['sym', 'fam']).size().unstack(); assert fam_ct.shape[1] == 8 and (fam_ct.min().min() >= 5), fam_ct
say('PASS T10 TEST-3 has events of all 8 families in both symbols (per symbol and family min ' + str(int(fam_ct.min().min())) + ')')

PASS T07 event hits equal a plain loop (first TEST-3 set, fusion, budget 2)
PASS T08 genuine episodes/day equal a plain loop (fusion, budget 2)
PASS T09 rolling thresholds are undefined (no alarm) before 1,440 bars of history; thresholds use the genuine series only
PASS T10 TEST-3 has events of all 8 families in both symbols (per symbol and family min 37)


## 7 · Results at the primary budget

In [ ]:
CALT = pd.read_csv(OUT/'fusion_v0_cal3_table.csv'); CALF = pd.read_csv(OUT/'fusion_v0_cal3_by_family.csv', index_col=0)
def cal_row(method, b): return CALT[(CALT.method == method) & (CALT.budget == b)].iloc[0]
WHO = {MODELS[0]: MODELS[0], MODELS[1]: MODELS[1], MODELS[2]: MODELS[2], 'union (3 models)': 'union', 'FUSION (Integrity Score)': 'Fusion'}
CALN = {MODELS[0]: MODELS[0], MODELS[1]: MODELS[1], MODELS[2]: MODELS[2], 'union (3 models)': 'union', 'FUSION (Integrity Score)': 'fusion'}
rows = []; HB = {}
bh = boot([hits(w, PRIM) for w in WHO.values()], ALLF)
for j, (lab, w) in enumerate(WHO.items()):
    h = hits(w, PRIM); cr = cal_row(CALN[lab], PRIM)
    rows.append({'method': lab, 'genuine eps/day TEST-B': gen_eps(w, PRIM), 'recall old 6': pct(macro_of(h, OLD)), 'recall new 2': pct(macro_of(h, NEW)), 'macro 8 TEST-3': pct(macro_of(h, ALLF)), '95% CI': ci(bh[:, j]), 'macro 8 CAL-3': pct(float(cr['macro 8'])), 'change': f'{100 * (macro_of(h, ALLF) - float(cr["macro 8"])):+.1f} pts'})
T_A = pd.DataFrame(rows); say('--- TEST-3 recall at the primary budget (2 genuine episodes/day), T1c, mean of BTC/ETH; CAL-3 for comparison ---'); say(T_A.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
rows = []
for b in BUD:
    for lab in ['FUSION (Integrity Score)', 'union (3 models)']:
        w = WHO[lab]; h = hits(w, b); cr = cal_row(CALN[lab], b); bb = boot([h], ALLF, n=max(200, NBOOT // 4))
        rows.append({'method': lab, 'budget': b, 'genuine eps/day TEST-B': gen_eps(w, b), 'genuine eps/day CAL': float(cr['CAL genuine eps/day']), 'recall old 6': pct(macro_of(h, OLD)), 'recall new 2': pct(macro_of(h, NEW)), 'macro 8 TEST-3': pct(macro_of(h, ALLF)), '95% CI': ci(bb[:, 0]), 'macro 8 CAL-3': pct(float(cr['macro 8']))})
T_B = pd.DataFrame(rows); say(''); say('--- budgets 1 / 2 / 4: fusion and the union ---'); say(T_B.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
FR = {lab: fam_rates(hits(w, PRIM)) for lab, w in WHO.items()}
T_C = pd.DataFrame({B.FAMS[f][:22]: {lab.split(' (')[0]: pct(FR[lab][f]) for lab in WHO} for f in ALLF}).T
T_C['FUSION CAL-3'] = [pct(float(CALF.loc[B.FAMS[f], 'fusion'])) if B.FAMS[f] in CALF.index else '' for f in ALLF]
say(''); say('--- TEST-3 recall per family, budget 2/day (all columns T1c; last column: fusion on CAL-3) ---'); say(T_C.to_string())

--- TEST-3 recall at the primary budget (2 genuine episodes/day), T1c, mean of BTC/ETH; CAL-3 for comparison ---
                  method  genuine eps/day TEST-B recall old 6 recall new 2 macro 8 TEST-3 95% CI macro 8 CAL-3   change
             AE-ctx fmax                    2.30        39.7%        33.8%          38.2%  35-42         35.5% +2.7 pts
              XGBoost v2                    1.50        69.7%        79.1%          72.0%  69-75         72.0% +0.0 pts
          Transformer v2                    1.67        74.3%        52.0%          68.7%  66-72         70.0% -1.3 pts
        union (3 models)                    4.89        81.8%        83.8%          82.3%  80-85         83.2% -1.0 pts
FUSION (Integrity Score)                    1.94        78.3%        76.4%          77.8%  75-80         77.8% +0.0 pts

--- budgets 1 / 2 / 4: fusion and the union ---
                  method  budget  genuine eps/day TEST-B  genuine eps/day CAL recall old 6 recall new 2 macro 8 TEST-3

## 8 · Pre-registered hypotheses

In [ ]:
HYP = {}
ge = {lab: gen_eps(w, PRIM) for lab, w in WHO.items()}
bounds = {lab: ge[lab] for lab in [MODELS[0], MODELS[1], MODELS[2], 'FUSION (Integrity Score)']}
HYP['H-T1'] = {'text': 'genuine episodes/day at budget 2 in [1, 4] for fusion and each model', 'values': {k: round(v, 2) for k, v in bounds.items()}, 'holds': bool(all(1.0 <= v <= 4.0 for v in bounds.values()))}
hF = hits('Fusion', PRIM); mF = macro_of(hF, ALLF); crF = float(cal_row('fusion', PRIM)['macro 8'])
HYP['H-T2'] = {'text': 'fusion macro-8 within 0.05 of CAL-3', 'test': round(mF, 4), 'cal': round(crF, 4), 'diff': round(mF - crF, 4), 'holds': bool(mF - crF >= -0.05)}
single = {m: macro_of(hits(m, PRIM), ALLF) for m in MODELS}; best = max(single, key=single.get); hB = hits(best, PRIM)
bb = boot([hF, hB], ALLF); dd = bb[:, 0] - bb[:, 1]; d_ = mF - single[best]
HYP['H-T3'] = {'text': 'fusion beats the strongest single model by >= 0.03, paired CI above 0', 'best_single': best, 'single_macro8': round(single[best], 4), 'fusion_macro8': round(mF, 4), 'diff': round(d_, 4), 'ci95': [round(float(np.percentile(dd, 2.5)), 4), round(float(np.percentile(dd, 97.5)), 4)], 'holds': bool(d_ >= 0.03 and np.percentile(dd, 2.5) > 0)}
hU = hits('union', PRIM); fo, uo = macro_of(hF, OLD), macro_of(hU, OLD)
HYP['H-T4'] = {'text': 'fusion old-6 recall >= union - 0.05 and fusion burden <= 60% of the union burden', 'fusion_old6': round(fo, 4), 'union_old6': round(uo, 4), 'fusion_eps': round(ge['FUSION (Integrity Score)'], 2), 'union_eps': round(ge['union (3 models)'], 2), 'holds': bool(fo >= uo - 0.05 and ge['FUSION (Integrity Score)'] <= 0.6 * ge['union (3 models)'])}
sn = {m: macro_of(hits(m, PRIM), NEW) for m in MODELS}; bn = max(sn, key=sn.get); fn_ = macro_of(hF, NEW)
HYP['H-T5'] = {'text': 'fusion new-2 recall >= best single - 0.05', 'fusion_new2': round(fn_, 4), 'best_single': bn, 'best_new2': round(sn[bn], 4), 'holds': bool(fn_ >= sn[bn] - 0.05)}
aT = aucs(lambda s: SC[('Fusion', s, 'r0')], lambda s: TLAB[(s, 0)], TEST_LO, AOFULL); aC = aucs(lambda s: SAVED[('Fusion', s, 'CAL_r0')], lambda s: CALLAB[s], CAL_LO, AOLEN)
HYP['H-T6'] = {'text': 'fusion PR-AUC (replica 0) >= CAL-3 - 0.05', 'test_pr_auc': round(float(aT[0]), 4), 'cal_pr_auc': round(float(aC[0]), 4), 'test_roc_auc': round(float(aT[1]), 4), 'cal_roc_auc': round(float(aC[1]), 4), 'holds': bool(aT[0] >= aC[0] - 0.05)}
say('--- pre-registered hypotheses (TEST-3, T1c, budget 2/day) ---')
for k, v in HYP.items(): say(f"{k} [{'HOLDS' if v['holds'] else 'FAILS'}] {v['text']}: " + json.dumps({a: b for a, b in v.items() if a not in ('text', 'holds')}))
say(''); say(f'ranking quality, replica 0 (positives = event bars of all 8 families; negatives = clean bars): fusion PR-AUC {pct(aT[0])} (CAL-3 {pct(aC[0])}), ROC-AUC {pct(aT[1])} (CAL-3 {pct(aC[1])})')
for m in MODELS:
    a1 = aucs(lambda s: SC[(m, s, 'r0')], lambda s: TLAB[(s, 0)], TEST_LO, AOFULL); a2 = aucs(lambda s: SAVED[(m, s, 'CAL_r0')], lambda s: CALLAB[s], CAL_LO, AOLEN); say(f'   {m}: PR-AUC {pct(a1[0])} (CAL-3 {pct(a2[0])}), ROC-AUC {pct(a1[1])} (CAL-3 {pct(a2[1])})')

--- pre-registered hypotheses (TEST-3, T1c, budget 2/day) ---
H-T1 [HOLDS] genuine episodes/day at budget 2 in [1, 4] for fusion and each model: {"values": {"AE-ctx fmax": 2.3, "XGBoost v2": 1.5, "Transformer v2": 1.67, "FUSION (Integrity Score)": 1.94}}
H-T2 [HOLDS] fusion macro-8 within 0.05 of CAL-3: {"test": 0.7779, "cal": 0.7775, "diff": 0.0004}
H-T3 [HOLDS] fusion beats the strongest single model by >= 0.03, paired CI above 0: {"best_single": "XGBoost v2", "single_macro8": 0.7202, "fusion_macro8": 0.7779, "diff": 0.0577, "ci95": [0.0328, 0.0826]}
H-T4 [HOLDS] fusion old-6 recall >= union - 0.05 and fusion burden <= 60% of the union burden: {"fusion_old6": 0.7827, "union_old6": 0.8179, "fusion_eps": 1.94, "union_eps": 4.89}
H-T5 [HOLDS] fusion new-2 recall >= best single - 0.05: {"fusion_new2": 0.7635, "best_single": "XGBoost v2", "best_new2": 0.7905}
H-T6 [HOLDS] fusion PR-AUC (replica 0) >= CAL-3 - 0.05: {"test_pr_auc": 0.6953, "cal_pr_auc": 0.7338, "test_roc_auc": 0.8635, "cal_

## 9 · Chance level, symbols, months, Watch tiers

In [ ]:
# chance level, per symbol, September versus October, Watch tiers
ch = chance('Fusion', PRIM, OLD), chance('Fusion', PRIM, NEW); say(f'chance level of the fusion alarm series (genuine alarms circularly shifted, 40 shifts): old 6 {pct(ch[0])}, new 2 {pct(ch[1])}')
rows = []
for s in SYMBOLS:
    k = (META.sym == s).to_numpy(); rows.append({'symbol': s, 'fusion macro 8': pct(macro_of(hF, ALLF, sel=k, syms=[s])), 'genuine eps/day': float(B.eps_per_day(gen_alarm('Fusion', s, PRIM), TV))})
say(''); say('--- fusion per symbol (budget 2) ---'); say(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.2f}'))
SEP = TV.copy(); SEP[TEST_LO + 30 * 1440:] = False; OCT = TV.copy(); OCT[:TEST_LO + 30 * 1440] = False
rows = [{'method': lab, 'Sep eps/day': gen_eps(w, PRIM, SEP), 'Oct eps/day': gen_eps(w, PRIM, OCT)} for lab, w in WHO.items()]
say(''); say('--- genuine alarm episodes/day, September versus October (budget 2) ---'); say(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.2f}'))
dl = np.mean([[B.episodes(gen_alarm('Fusion', s, PRIM)[TEST_LO + d * 1440:TEST_LO + (d + 1) * 1440]) for d in range(61)] for s in SYMBOLS], axis=0)
say(f'fusion alert episodes per genuine day: median {np.median(dl):.1f}, mean {dl.mean():.2f}, max {dl.max():.1f}, days above 4: {int((dl > 4).sum())} of 61')
rows = []
for lab, w in [('Alert (2/day)', 'Fusion'), ('Watch-4', ('watch', 4)), ('Watch-8', ('watch', 8))]:
    b = {'Alert (2/day)': 2, 'Watch-4': 4, 'Watch-8': 8}[lab]; h = hits(w if lab != 'Alert (2/day)' else 'Fusion', b if lab != 'Alert (2/day)' else PRIM); cr_ = None
    rows.append({'tier': lab, 'genuine eps/day TEST-B': gen_eps(w if lab != 'Alert (2/day)' else 'Fusion', b if lab != 'Alert (2/day)' else PRIM), 'macro 8': pct(macro_of(h, ALLF)), 'volume_burst': pct(float(fam_rates(h)[3])), 'recall old 6': pct(macro_of(h, OLD)), 'recall new 2': pct(macro_of(h, NEW))})
T_W = pd.DataFrame(rows); say(''); say('--- Watch / Alert tiers on TEST (fusion, T1c; thresholds from notebook 40) ---'); say(T_W.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

chance level of the fusion alarm series (genuine alarms circularly shifted, 40 shifts): old 6 3.1%, new 2 6.5%

--- fusion per symbol (budget 2) ---
 symbol fusion macro 8  genuine eps/day
BTCUSDT          81.1%             2.11
ETHUSDT          74.5%             1.77

--- genuine alarm episodes/day, September versus October (budget 2) ---
                  method  Sep eps/day  Oct eps/day
             AE-ctx fmax         2.22         2.39
              XGBoost v2         1.50         1.50
          Transformer v2         1.72         1.63
        union (3 models)         4.75         5.02
FUSION (Integrity Score)         2.13         1.76
fusion alert episodes per genuine day: median 1.5, mean 1.94, max 5.5, days above 4: 6 of 61

--- Watch / Alert tiers on TEST (fusion, T1c; thresholds from notebook 40) ---
         tier  genuine eps/day TEST-B macro 8 volume_burst recall old 6 recall new 2
Alert (2/day)                    1.94   77.8%        21.1%        78.3%        76.4%
      Wat

## 10 · Save, guard, share block

In [ ]:
T_A.to_csv(OUT/'test_b_recall_budget2.csv', index=False); T_B.to_csv(OUT/'test_b_recall_budgets.csv', index=False); T_C.to_csv(OUT/'test_b_by_family.csv'); T_W.to_csv(OUT/'test_b_watch_tiers.csv', index=False)
wfiles = sorted(str(p.relative_to(OUT)) for sub in ['ae', 'xgb', 'transformer'] for p in (OUT/sub).glob('*') if p.suffix in ('.pt', '.json') and not p.name.endswith('.info.json') and p.stat().st_size > 0)
RESJ = {'opened_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'protocol_T0_sha256': PH, 'protocol_B2_sha256': ph, 'library_sha16': sha_file('c03_infer.py'), 'b2_lib_sha16': sha_file(OUT/'b2_lib.py'),
        'declared_sha16': {f: sha_file(OUT/f) for f in ['declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json']}, 'stacker_sha16': sha_file(FUS/'fusion_v0_stacker.joblib'), 'weights_sha16': {f: sha_file(OUT/f) for f in wfiles},
        'test_score_files_sha16': {s: sha_file(OUT/'scores'/f'test_b_{s}.npz') for s in SYMBOLS}, 'hypotheses': HYP, 'genuine_eps_day_budget2': {k: round(v, 3) for k, v in ge.items()},
        'macro8_budget2': {lab: round(macro_of(hits(w, PRIM), ALLF), 4) for lab, w in WHO.items()}, 'mock': MOCK}
(OUT/'test_b_results_v1.json').write_text(json.dumps(RESJ, indent=1, default=str))
say(''); say('--- reading (pre-registered; no change after TEST) ---')
nh = sum(v['holds'] for v in HYP.values()); say(f'{nh} of {len(HYP)} pre-registered hypotheses hold. Failed ones are reported as failed and nothing is retuned: any improvement becomes a new version tested on later data.')
say('Scores of TEST-B and TEST-3 are saved (scores/test_b_*.npz) so that notebook 32b (v1 re-scoring, H-B6), explanation and uncertainty checks can run without touching the v2 models again.')
block = '\n'.join(LINES); (OUT/'b2_step17_share_block.txt').write_text(block)
GUARD.write_text(json.dumps({'opened_utc': RESJ['opened_utc'], 'note': 'TEST-B / TEST-3 opened once by notebook 32. Delete this file only if you intend to void the TEST claim.', 'results_file': 'test_b_results_v1.json', 'share_block_sha16': hashlib.sha256(block.encode()).hexdigest()[:16]}, indent=1))
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step17_share_block.txt, or copy everything below) =====\n' + block[-16000:])


--- reading (pre-registered; no change after TEST) ---
6 of 6 pre-registered hypotheses hold. Failed ones are reported as failed and nothing is retuned: any improvement becomes a new version tested on later data.
Scores of TEST-B and TEST-3 are saved (scores/test_b_*.npz) so that notebook 32b (v1 re-scoring, H-B6), explanation and uncertainty checks can run without touching the v2 models again.

===== SHARE BLOCK (attach models_b2_v1/b2_step17_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; test protocol T0 sha256 876d176c01c0712a; device cpu
inference library sha256 3b5d9df23521adf9; b2_lib sha256 1b0b46630763c944
frozen weights loaded: AE-ctx (3 seeds, N1, mean10), XGBoost A|pooled|d3 (N0), Transformer W16_d32_L1 (N1); stacker LR C=0.01
--- parity: library re-scoring versus the scores saved by notebooks 28-31 / 41 (August genuine and CAL-3 replica 0) ---
    sym          model series  same NaN pattern  max abs diff  median abs diff  spea